# L0dfiDoTA on DAD + figures K5dfiDoTA ("LaGoVAD + KIM") vs L0dfiDoTA ("LaGoVAD") on DoTA, DADA, DAD

Follows §58 (`KIM_RUN.md`). §58 read L0dfiDoTA on DoTA-CAP-dev and DADA only; the figures need its **DAD** read too. One `core.evaluate` on the plain CLIP cache `DAD_s7_ncc` (§36 R1 input, checked equal to the CLIP columns of the bake K5dfiDoTA read DAD on), then `zs_curves` on three sets and the DAD placebo (DAD macro is never read without it, §56 DA5). **Printed, decides nothing; K5f stays.**

| step | what | GPU | estimate |
|---|---|---|---|
| 1-4 | setup, deps, helpers, preflight | no | 2-3 min |
| 5 | same CLIP frames on DAD (plain vs K5dfiDoTA's bake) | no | 2-5 min |
| 6 | L0dfiDoTA on DAD (skipped if on Drive) | **yes** | 5-10 min |
| 7 | figures (onset + grids, DoTA in-domain, no n) + DAD placebo -> **paste the printed block to Claude** | no | 2-3 min |

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, §55/§58 rows, DAD data and caches; every output here
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '--hide-n' in (REPO / 'core/tools/zs_curves.py').read_text(), 'zs_curves has no --hide-n -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5, SEED = constants.KIM_MAIN_ARM, constants.KIM_SEED
CTRL, L0 = constants.KIM_DOTA_TRAIN_ROW, constants.KIM_DOTA_TRAIN_L0_ROW
DAD_NAME, DAD_STRIDE, DAD_SET = constants.DAD_DATASET, constants.DAD_STRIDE, constants.DAD_OUTPUT_SET
NEX, DST_SUF = constants.NEXAR_STATS_BAKE_SUFFIX, constants.DOTA_TRAIN_STATS_BAKE_SUFFIX
ROOT     = constants.OUTPUT_ROOT / constants.KIM_DOTA_TRAIN_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
DAD_PLAIN = constants.CLIP_CACHE_DIR / f'{DAD_NAME}_s{DAD_STRIDE}_ncc'
DAD_BAKE = constants.CACHE_ROOT / 'v2' / 'A2_S' / (f'{DAD_NAME}_s{DAD_STRIDE}_nexar'[: -len(NEX)] + DST_SUF)   # §55's restat
FIG_TAG  = 'k5dfidota_vs_l0dfidota_vs_lagovad_s2024'
FIGURES  = (constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME / 'REPORTS' / constants.ZS_CURVES_FIGURES_DIRNAME
            / 'train_dota' / 'dota_dada_dad' / FIG_TAG)
LOCAL    = Path('/content/l0dfidota_dad')
LOCAL.mkdir(parents=True, exist_ok=True)
KIM_LABEL, LAG_LABEL, PUB_LABEL = 'LaGoVAD + KIM', 'LaGoVAD', 'LaGoVAD (public checkpoint)'
LAG     = constants.ZS_LAGOVAD_ARM
PUB_ROOT = constants.OUTPUT_ROOT / constants.ZS_LAGOVAD_OUTPUT_DIRNAME / 'eval'
PUB = {'dota': PUB_ROOT / 'dota' / LAG / 'B' / 'clip_scores.npz', 'dada': PUB_ROOT / 'dada' / LAG / 'clip_scores.npz',
       'dad': PUB_ROOT / DAD_SET / LAG / constants.ZS_LAGOVAD_RUN}   # released checkpoint: onset only


def row_paths(row):
    run_dir = ROOT / row / K5 / f's{SEED}'
    return run_dir, run_dir / 'checkpoint_last.pt', ROOT / row / 'eval'


EV = {r: row_paths(r)[2] for r in (CTRL, L0)}
READS = {(s, r): EV[r] / s / K5 / 'clip_scores.npz' for s in ('dota', 'dada') for r in (CTRL, L0)}
INPUTS = (('K5dfiDoTA DAD read', EV[CTRL] / DAD_SET / K5 / f's{SEED}' / 'scores'),
          ('L0dfiDoTA checkpoint', row_paths(L0)[1]), ('L0dfiDoTA verified', row_paths(L0)[0] / 'verified.json'),
          ('L0dfiDoTA config', row_paths(L0)[0] / 'config.yaml'),
          ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME), ('DAD plain CLIP', DAD_PLAIN),
          ('DAD bake (DoTA statistics)', DAD_BAKE / constants.V2_INPUT_MANIFEST_FILENAME),
          *[(f'{r} {s} read', p) for (s, r), p in READS.items()],
          *[(f'public checkpoint {s} read', p) for s, p in PUB.items()])
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing (run k5dfidota_ablation.ipynb first?) -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | figures -> {FIGURES}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 6 needs one (skipped if the read is on Drive)'

In [ ]:
import numpy as np  # noqa: E402
import torch  # noqa: E402
import yaml  # noqa: E402

from core.config import Config                       # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.dota_train_extract import train_clips  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    names = [f.name for f in src.iterdir() if f.is_file()] if ids is None else \
        [f'{v}.npy' for v in ids] + [m.name for p in ('*.json', '*.npz') for m in src.glob(p)]
    todo = [n for n in names if not (dst / n).exists()]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda n: shutil.copy2(src / n, dst / n), todo))
    print(f'  staged {src.name}: {len(todo)} files in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def on_drive(folder, min_bytes=1024):
    """Ids of every finished ``.npy`` in ``folder`` from ONE listing (save_array = tmp + rename)."""
    if not folder.is_dir():
        return set()
    with os.scandir(folder) as entries:
        return {e.name[:-4] for e in entries if e.name.endswith('.npy') and e.stat().st_size >= min_bytes}


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(run_dir, steps):
    rec, ckpt = run_dir / 'verified.json', run_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


DEFAULTS = _flat(Config().to_dict())


def config_diff(base, got):
    """Keys that differ; a key added after ``base`` trained may sit at its default."""
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k):
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out

l0_cfg = _flat(yaml.safe_load((row_paths(L0)[0] / 'config.yaml').read_text()))
assert not l0_cfg['kim.enabled'] and l0_cfg['v2.motion'] == constants.V2_OFF and l0_cfg['data.dataset'] == constants.DOTA_TRAIN_DATASET, \
    'not L0dfiDoTA'
STEPS = json.loads((row_paths(L0)[0] / 'verified.json').read_text())['step']
assert ckpt_step(row_paths(L0)[1]) == STEPS == metrics_rows(row_paths(L0)[0] / 'metrics.jsonl')[-1]['global_step'] == 1764, 'L0dfiDoTA: J10'
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(DAD_IDS) == constants.DAD_TEST_VIDEOS, len(DAD_IDS)
print(f'L0dfiDoTA step {STEPS} (J10) | DAD {len(DAD_IDS)} videos')

## 5. Same CLIP frames on DAD

In [ ]:
L_PLAIN = stage(DAD_PLAIN, LOCAL / 'stage' / 'dad_plain', DAD_IDS)
L_BAKE = stage(DAD_BAKE, LOCAL / 'stage' / 'dad_bake', DAD_IDS)
run([sys.executable, '-m', 'core.tools.zs_lagovad_readout', 'check-inputs', '--plain', L_PLAIN, '--bake', L_BAKE,
     '--ids-file', write_ids(LOCAL / 'ids.txt', DAD_IDS), '--plain-stride', 1])
shutil.rmtree(L_BAKE)
print('same CLIP frames on DAD: plain == K5dfiDoTA bake CLIP columns')

## 6. L0dfiDoTA on DAD (`core.evaluate`, plain CLIP s7, raw scores)

In [ ]:
out = EV[L0] / DAD_SET / K5 / f's{SEED}'
if not (out / 'results.json').exists():
    assert torch.cuda.is_available(), 'torch sees no GPU'
    local = LOCAL / 'eval' / 'dad'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.evaluate', '--ckpt', row_paths(L0)[1], '--set', f'data.dataset={DAD_NAME}',
         '--data-dir', stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta'), '--clip-dir', L_PLAIN,
         '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / 'dad.log')
    publish(local, out)
    flush_drive()
r = json.loads((out / 'results.json').read_text())
assert r['score_norm'] == 'none' and r['num_videos'] == len(DAD_IDS), r
print(f"{L0} DAD: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")

## 7. Figures + DAD placebo -> **paste the printed block to Claude**

In [ ]:
from IPython.display import Image as Show, display  # noqa: E402

local = LOCAL / 'figures'
shutil.rmtree(local, ignore_errors=True)
DAD_RUN = {r: EV[r] / DAD_SET / K5 / f's{SEED}' for r in (CTRL, L0)}
LABEL = {CTRL: KIM_LABEL, L0: LAG_LABEL}
reads = [a for (s, r), p in READS.items() for a in ('--read', s, LABEL[r], p)]
reads += [a for r, p in DAD_RUN.items() for a in ('--read', 'dad', LABEL[r], p)]
run([sys.executable, '-m', 'core.tools.zs_curves', '--rank-by', KIM_LABEL, '--name', FIG_TAG,
     '--seed-label', f'K5dfiDoTA and L0dfiDoTA trained on DoTA, s{SEED}, 1 seed', '--pick-auc', 'global', '--set-title', 'dota', 'DoTA',
     '--set-domain', 'dota', constants.ZS_CURVES_DOMAIN_IN, '--hide-n', *reads, '--out-dir', local])
# onset with a third line, the released checkpoint; grids stay two-model (user)
onset3 = LOCAL / 'onset3'
shutil.rmtree(onset3, ignore_errors=True)
pub = [a for s, p in PUB.items() for a in ('--read', s, PUB_LABEL, p)]
run([sys.executable, '-m', 'core.tools.zs_curves', '--rank-by', KIM_LABEL, '--name', FIG_TAG.replace('_s2024', '_onset_s2024'),
     '--seed-label', f'K5dfiDoTA and L0dfiDoTA trained on DoTA, s{SEED}, 1 seed', '--pick-auc', 'global',
     '--set-title', 'dota', 'DoTA', '--set-domain', 'dota', constants.ZS_CURVES_DOMAIN_IN, '--hide-n', *reads, *pub,
     '--out-dir', onset3])
shutil.copy2(onset3 / 'onset_aligned.png', local / 'onset_aligned.png')
shutil.copy2(onset3 / f"{constants.ZS_CURVES_STEM}_{FIG_TAG.replace('_s2024', '_onset_s2024')}.json", local)
publish(local, FIGURES)

# DAD placebo beside the DAD macro (§56 DA5): control K5dfiDoTA
diag = LOCAL / 'dad_position'
run([sys.executable, '-m', 'core.tools.dad_position_diag', '--row', CTRL, DAD_RUN[CTRL], '--row', L0, DAD_RUN[L0],
     '--control', CTRL, '--out-dir', diag])
publish(diag, REPORTS / 'dad_position_l0dfidota')
flush_drive()
for name in ('onset_aligned.png', 'grid_dota.png', 'grid_dada.png', 'grid_dad.png'):
    display(Show(filename=str(FIGURES / name)))
print((diag / 'dad_position_diag.md').read_text())
drive.flush_and_unmount()
print('Drive flushed -- paste the DAD line of step 6 and the placebo table above to Claude')